# Dmitri Gomon, i.d. 304442353

In [52]:
import torch
import pytest



# A

In [53]:
def _can_expand_to(source: torch.Tensor, target: torch.Tensor) -> bool:
    """Check whether a source tensor can be expanded to the shape of a target tensor.

    Follows PyTorch broadcasting semantics: the source tensor must have fewer
    than or equal dimensions compared to the target, and trailing dimensions
    must either match the target's size or equal 1.

    Args:
        source: Tensor whose shape is evaluated for expansion.
        target: Target tensor providing the desired output shape.

    Returns:
        bool: True if source.expand_as(target) is valid, False otherwise.
    """
    index_A = source.dim() - 1
    index_B = target.dim() - 1

    if index_A > index_B:
        # A has more dimensions than B
        return False

    while index_A >= 0:
        if source.shape[index_A] != 1 and source.shape[index_A] != target.shape[index_B]:
            # Current A's dimension can only be expanded if its length is 1 or its length matches that of B's
            return False

        index_A -= 1
        index_B -= 1
    return True


def my_expand_as(input_tensor: torch.Tensor, other: torch.Tensor) -> torch.Tensor:
    r"""Expand `input` to the same size as `other`.

        Equivalent to calling an explicit tensor expansion along singleton
        dimensions to match `other.size()`.

        Args:
            input_tensor (Tensor): the input tensor.
            other (Tensor): the tensor whose size should be matched.

        Returns:
            Tensor: A new tensor matching `other.shape`.

        Raises:
            RuntimeError: If `input_tensor` cannot be expanded to the size of `other`.
        """
    if not _can_expand_to(input_tensor, other):
        raise RuntimeError

    # Clone to avoid in-place modification of input
    c = input_tensor.clone()

    # Add singleton dimensions at the beginning until ranks match
    while c.dim() < other.dim():
        c = c.unsqueeze(0)

    for dim in range(c.dim()):
        if c.shape[dim] == 1 and other.shape[dim] > 1:
            # Replicate along this axis by concatenating copies
            c = torch.cat([c] * other.shape[dim], dim=dim)

    return c


# B

In [54]:
def can_be_broadcasted(a: torch.Tensor, b: torch.Tensor) -> tuple[bool, torch.Size | None]:
    r"""Checks whether two tensors are broadcastable according to PyTorch rules.

        Args:
            a: First input tensor.
            b: Second input tensor.

        Returns:
            A tuple `(is_broadcastable, broadcast_shape)`:
            - `is_broadcastable`: `True` if tensors can be broadcast together, `False` otherwise.
            - `broadcast_shape`: The resulting `torch.Size` if broadcastable, or `None`.
        """

    c = [0] * max(len(a.shape), len(b.shape))

    for i in range(1, min(a.dim(), b.dim()) + 1):
        dim_a, dim_b = a.shape[-i], b.shape[-i]
        if (dim_a != dim_b) and (dim_a != 1) and (dim_b != 1):
            return False, None
        # b may be 0 here, thus not using max()
        c[-i] = dim_a if dim_b == 1 else dim_b

    diff = a.dim() - b.dim()
    if diff != 0:
        longer = a.shape if diff > 0 else b.shape
        c[:abs(diff)] = longer[:abs(diff)]

    return True, torch.Size(c)

# C

In [55]:
def _broadcast_to(t: torch.Tensor, target_shape: torch.Size) -> torch.Tensor:
    """Explicitly broadcast a tensor to a target shape using repetition.

        Aligns tensor rank by prepending singleton dimensions, then replicates
        singleton dimensions via concatenation to match the specified shape.

        Args:
            t: Input tensor to broadcast.
            target_shape: Destination shape. Each dimension must be compatible
                with broadcasting rules (equal to the current dimension or current is 1).

        Returns:
            A new tensor broadcasted to `target_shape`.
        """
    res = t.clone()

    # Step 1: Align ranks by prepending singleton dimensions
    while res.dim() < len(target_shape):
        res = res.unsqueeze(0)

    # Step 2: Expand each dimension using torch.cat
    for d, target_size in enumerate(target_shape):
        curr_size = res.shape[d]
        if curr_size == 1 and target_size > 1:
            # Replicate along dimension d
            res = torch.cat([res] * target_size, dim=d)
        elif curr_size == 1 and target_size == 0:
            # Handle size 0 edge case: reshaping empty tensor to target slice
            empty_shape = list(res.shape)
            empty_shape[d] = 0
            res = torch.empty(empty_shape, dtype=res.dtype, device=res.device)
    return res


def my_broadcast(a: torch.Tensor, b: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Broadcast two tensors to a common compatible shape.

        Args:
            a: First input tensor.
            b: Second input tensor.

        Returns:
            tuple[torch.Tensor, torch.Tensor]: A tuple containing both tensors
            broadcasted to the shared target shape.

        Raises:
            RuntimeError: If shapes of `a` and `b` are incompatible for broadcasting.
        """
    possible, size_c = can_be_broadcasted(a, b)

    if not possible:
        raise RuntimeError(f"The size of tensor A {a.shape} must match the size of tensor B {b.shape}")

    result_a = _broadcast_to(a, size_c)
    result_b = _broadcast_to(b, size_c)

    return result_a, result_b


# D

## testing can_be_expanded()

In [56]:
import ipytest
ipytest.config(
    clean=True,
    run_in_thread=True,
    coverage=False,  # Prevents passing --cov arguments to pytest
)

VALID_EXPAND_CASES = [
    # (input_tensor, target_tensor, case_id)
    (
        torch.tensor([1]),
        torch.tensor([3, 2]),
        "1d_singleton_to_1d",
    ),
    (
        torch.tensor([5, 7]),
        torch.tensor([3, 2]),
        "1d_same_shape",
    ),
    (
        torch.tensor([5, 7]),
        torch.tensor([[1, 2], [3, 4]]),
        "1d_to_2d_row_broadcast",
    ),
    (
        torch.tensor([1]),
        torch.tensor([[3, 2], [4, 6]]),
        "1d_singleton_to_2d",
    ),
    (
        torch.tensor([97, 17, 2]),
        torch.tensor([[11, 12, 13], [21, 22, 23], [31, 32, 33]]),
        "1d_to_square_2d",
    ),
    # --- Augmented Cases ---
    (
        torch.tensor(42),
        torch.tensor([[1, 2], [3, 4]]),
        "0d_scalar_to_2d",
    ),
    (
        torch.tensor([[1], [2]]),
        torch.tensor([[0, 0, 0], [0, 0, 0]]),
        "2d_column_singleton_to_2d",
    ),
    (
        torch.tensor([1, 2]),
        torch.empty(2, 2, 2),
        "1d_to_3d_expansion",
    ),
]

INVALID_EXPAND_CASES = [
    # Incompatible trailing dimension (2 != 3)
    (
        torch.tensor([1, 2]),
        torch.tensor([1, 2, 3]),
        "mismatched_trailing_dim",
    ),
    # Attempting to expand a non-singleton dimension (2 -> 3)
    (
        torch.tensor([[1, 2], [3, 4]]),
        torch.empty(3, 2),
        "incompatible_leading_dim",
    ),
    # Input has more dimensions than target
    (
        torch.tensor([[1, 2]]),
        torch.tensor([1, 2]),
        "input_higher_dim_than_target",
    ),
]

ALL_EXPAND_CASES = [
                       (*case[:2], True, case[2]) for case in VALID_EXPAND_CASES
                   ] + [
                       (*case[:2], False, case[2]) for case in INVALID_EXPAND_CASES
                   ]


@pytest.mark.parametrize(
    "source, target",
    [case[:2] for case in ALL_EXPAND_CASES],
    ids=[case[2] for case in ALL_EXPAND_CASES],
)
def test_can_be_expanded(
        source: torch.Tensor, target: torch.Tensor,
) -> None:
    actual = _can_expand_to(source, target)
    if actual:
        # Must NOT raise an exception when expandable
        try:
            my_expand_as(source, target)
        except RuntimeError as exc:
            pytest.fail(f"can_be_expanded returned True, but my_expand_as raised: {exc}")
    else:
        # MUST raise RuntimeError when not expandable
        with pytest.raises(RuntimeError):
            my_expand_as(source, target)

ipytest.run("-v")

======================================= test session starts ========================================
platform darwin -- Python 3.13.18, pytest-9.2.0, pluggy-1.7.0 -- /Users/dgomon/miniconda3/envs/deep_learning/bin/python
cachedir: .pytest_cache
rootdir: /Users/dgomon/Documents/OpenU/deep_learning/assignments/maman11
plugins: platformdirs-4.14.0, anyio-4.15.1
collecting ... collected 44 items

t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True0] PASSED                    [  2%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True1] PASSED                    [  4%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True2] PASSED                    [  6%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True3] PASSED                    [  9%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True4] PASSED                    [ 11%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True5] PASSED                    [ 13%]
t_5f

<ExitCode.OK: 0>

## testing my_expand_as()

In [57]:
import ipytest
ipytest.config(
    clean=True,
    run_in_thread=True,
    coverage=False,  # Prevents passing --cov arguments to pytest
)


VALID_EXPAND_CASES = [
    # (input_tensor, target_tensor, expected_output, case_id)
    (
        torch.tensor([1]),
        torch.tensor([3, 2]),
        torch.tensor([1, 1]),
        "1d_singleton_to_1d",
    ),
    (
        torch.tensor([5, 7]),
        torch.tensor([3, 2]),
        torch.tensor([5, 7]),
        "1d_same_shape",
    ),
    (
        torch.tensor([5, 7]),
        torch.tensor([[1, 2], [3, 4]]),
        torch.tensor([[5, 7], [5, 7]]),
        "1d_to_2d_row_broadcast",
    ),
    (
        torch.tensor([1]),
        torch.tensor([[3, 2], [4, 6]]),
        torch.tensor([[1, 1], [1, 1]]),
        "1d_singleton_to_2d",
    ),
    (
        torch.tensor([97, 17, 2]),
        torch.tensor([[11, 12, 13], [21, 22, 23], [31, 32, 33]]),
        torch.tensor([[97, 17, 2], [97, 17, 2], [97, 17, 2]]),
        "1d_to_square_2d",
    ),
    # --- Augmented Cases ---
    (
        torch.tensor(42),
        torch.tensor([[1, 2], [3, 4]]),
        torch.tensor([[42, 42], [42, 42]]),
        "0d_scalar_to_2d",
    ),
    (
        torch.tensor([[1], [2]]),
        torch.tensor([[0, 0, 0], [0, 0, 0]]),
        torch.tensor([[1, 1, 1], [2, 2, 2]]),
        "2d_column_singleton_to_2d",
    ),
    (
        torch.tensor([1, 2]),
        torch.empty(2, 2, 2),
        torch.tensor([[[1, 2], [1, 2]], [[1, 2], [1, 2]]]),
        "1d_to_3d_expansion",
    ),
]


INVALID_EXPAND_CASES = [
    # Incompatible trailing dimension (2 != 3)
    (
        torch.tensor([1, 2]),
        torch.tensor([1, 2, 3]),
        "mismatched_trailing_dim",
    ),
    # Attempting to expand a non-singleton dimension (2 -> 3)
    (
        torch.tensor([[1, 2], [3, 4]]),
        torch.empty(3, 2),
        "incompatible_leading_dim",
    ),
    # Input has more dimensions than target
    (
        torch.tensor([[1, 2]]),
        torch.tensor([1, 2]),
        "input_higher_dim_than_target",
    ),
]


@pytest.mark.parametrize(
    "source, target, expected",
    [case[:3] for case in VALID_EXPAND_CASES],
    ids=[case[3] for case in VALID_EXPAND_CASES],
)
def test_my_expand_as_valid(source: torch.Tensor, target: torch.Tensor, expected: torch.Tensor):
    result = my_expand_as(source, target)

    assert result.shape == target.shape
    torch.testing.assert_close(result, expected)


@pytest.mark.parametrize(
    "source, target",
    [case[:2] for case in INVALID_EXPAND_CASES],
    ids=[case[2] for case in INVALID_EXPAND_CASES],
)
def test_my_expand_as_incompatible_shapes(source: torch.Tensor, target: torch.Tensor):
    with pytest.raises(RuntimeError):
        my_expand_as(source, target)

ipytest.run("-v")

======================================= test session starts ========================================
platform darwin -- Python 3.13.18, pytest-9.2.0, pluggy-1.7.0 -- /Users/dgomon/miniconda3/envs/deep_learning/bin/python
cachedir: .pytest_cache
rootdir: /Users/dgomon/Documents/OpenU/deep_learning/assignments/maman11
plugins: platformdirs-4.14.0, anyio-4.15.1
collecting ... collected 44 items

t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True0] PASSED                    [  2%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True1] PASSED                    [  4%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True2] PASSED                    [  6%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True3] PASSED                    [  9%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True4] PASSED                    [ 11%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True5] PASSED                    [ 13%]
t_5f

<ExitCode.OK: 0>

## testing my_broadcast()

In [58]:
import ipytest
ipytest.config(
    clean=True,
    run_in_thread=True,
    coverage=False,  # Prevents passing --cov arguments to pytest
)

VALID_BROADCAST_CASES = [
    # (tensor A, tensor B, case_id)
    (
        torch.tensor([1]),
        torch.tensor([[11, 12, 13], [21, 22, 23]]),
        "1d_scalar_like_with_2d",
    ),
    (
        torch.tensor(5),
        torch.tensor([1, 2, 3]),
        "0d_scalar_with_1d",
    ),
    (
        torch.tensor(7),
        torch.tensor(8),
        "two_0d_scalars",
    ),
    (
        torch.tensor([[1, 2], [3, 4]]),
        torch.tensor([[5, 6], [7, 8]]),
        "identical_shapes",
    ),
    (
        torch.tensor([[1, 2, 3]]),
        torch.tensor([[4], [5]]),
        "row_and_column_vectors",
    ),
    (
        torch.tensor([10, 20, 30]),
        torch.tensor([[1, 2, 3], [4, 5, 6]]),
        "rank_extension_1d_with_2d",
    ),
    (
        torch.arange(6).reshape(2, 1, 3),
        torch.arange(4).reshape(1, 4, 1),
        "higher_dimensional_expansion",
    ),
]

INVALID_BROADCAST_CASES = [
    (
        torch.tensor([1, 2]),
        torch.tensor([1, 2, 3]),
        "incompatible_trailing_dims",
    ),
    (
        torch.zeros(2, 3),
        torch.zeros(2, 4),
        "incompatible_inner_dims",
    ),
    (
        torch.zeros(3, 2),
        torch.zeros(4, 2),
        "incompatible_leading_dims",
    ),
    (
        torch.zeros(2, 3, 4),
        torch.zeros(2, 5, 4),
        "multidimensional_mismatch",
    ),
]

ALL_BROADCAST_CASES = [
                          (*case[:2], True, case[2]) for case in VALID_BROADCAST_CASES
                      ] + [
                          (*case[:2], False, case[2]) for case in INVALID_BROADCAST_CASES
                      ]


@pytest.mark.parametrize(
    "a, b",
    [case[:2] for case in ALL_BROADCAST_CASES],
    ids=[case[2] for case in ALL_BROADCAST_CASES],
)
def test_can_be_broadcasted(
        a: torch.Tensor, b: torch.Tensor
) -> None:
    (actual_are_compatible, actual_shape) = can_be_broadcasted(a, b)
    if actual_are_compatible:
        # Must NOT raise an exception when can be broadcasted
        try:
            torch.broadcast_tensors(a, b)
        except RuntimeError as exc:
            pytest.fail(f"can_be_broadcasted returned True, but torch.broadcast_tensors raised: {exc}")
    else:
        # MUST raise RuntimeError when not expandable
        with pytest.raises(RuntimeError):
            torch.broadcast_tensors(a, b)


@pytest.mark.parametrize(
    "a, b",
    [case[:2] for case in VALID_BROADCAST_CASES],
    ids=[case[2] for case in VALID_BROADCAST_CASES],
)
def test_my_broadcast_valid_cases(a: torch.Tensor, b: torch.Tensor) -> None:
    expected_a, expected_b = torch.broadcast_tensors(a, b)
    actual_a, actual_b = my_broadcast(a, b)

    torch.testing.assert_close(actual_a, expected_a)
    torch.testing.assert_close(actual_b, expected_b)


@pytest.mark.parametrize(
    "a, b",
    [case[:2] for case in INVALID_BROADCAST_CASES],
    ids=[case[2] for case in INVALID_BROADCAST_CASES],
)
def test_my_broadcast_invalid_cases(a: torch.Tensor, b: torch.Tensor) -> None:
    with pytest.raises((RuntimeError, ValueError)):
        my_broadcast(a, b)


ipytest.run("-v")

======================================= test session starts ========================================
platform darwin -- Python 3.13.18, pytest-9.2.0, pluggy-1.7.0 -- /Users/dgomon/miniconda3/envs/deep_learning/bin/python
cachedir: .pytest_cache
rootdir: /Users/dgomon/Documents/OpenU/deep_learning/assignments/maman11
plugins: platformdirs-4.14.0, anyio-4.15.1
collecting ... collected 44 items

t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True0] PASSED                    [  2%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True1] PASSED                    [  4%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True2] PASSED                    [  6%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True3] PASSED                    [  9%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True4] PASSED                    [ 11%]
t_5fb5adb0d7394f46bc0dc97eb7e9b8e0.py::test_can_be_expanded[True5] PASSED                    [ 13%]
t_5f

<ExitCode.OK: 0>